# 2 · Inferencia y predicción

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decostruttivismo/IIC3800-2026-NCC-IA/blob/main/clase-24-agosto/notebooks/02_prediccion.ipynb)

**Modelos de Aprendizaje en Neurociencia Cognitiva e Inteligencia Artificial**  
Clase del 24 de agosto · Interferencia cognitiva (Stroop)

---

### La pregunta

En el cuaderno 1 establecimos que la incongruencia cuesta 65 ms, con
$p \approx 10^{-110}$. Eso es **inferencia**: cuán compatible es un efecto con el azar
bajo un modelo determinado.

Aquí hacemos otra pregunta:

> Con la condición, el número de ensayo y la edad, **¿con qué exactitud puedo predecir
> el RT de un participante que el modelo nunca ha visto?**

Es **predicción**: en qué medida el modelo generaliza a observaciones nuevas.

Son preguntas complementarias, no rivales, y —esto es lo que veremos— sus respuestas
pueden diferir muchísimo sobre exactamente los mismos datos y el mismo modelo.

## 1. Cargar los datos

El CSV se lee por URL desde el repositorio del curso: no hay que subir nada.

In [ ]:
# --- Configuración: de dónde se leen los datos -------------------------------
USUARIO = "decostruttivismo"
REPO    = "IIC3800-2026-NCC-IA"
RAMA    = "main"
CARPETA = "clase-24-agosto"
ARCHIVO = "datos/clase_24agosto_dataset.csv"

URL = f"https://raw.githubusercontent.com/{USUARIO}/{REPO}/{RAMA}/{CARPETA}/{ARCHIVO}"

import os
CANDIDATOS = [ARCHIVO,
              os.path.join("..", ARCHIVO),
              os.path.join("..", "..", ARCHIVO),
              os.path.basename(ARCHIVO)]
FUENTE = next((p for p in CANDIDATOS if os.path.exists(p)), URL)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv(FUENTE)
print("Leyendo desde:", FUENTE)
print("Filas y columnas:", data.shape)
data.head()

## 2. La decisión que define el problema: cómo se parte la muestra

Para medir generalización hay que evaluar sobre datos que el modelo no usó para
aprender. Pero **cómo** se separan esos datos no es un detalle técnico: define qué
pregunta se está respondiendo.

| forma de partir | qué mide |
|---|---|
| al azar por ensayo | «¿puedo predecir un ensayo nuevo **de alguien que ya conozco**?» |
| por participante | «¿puedo predecir a **una persona nueva**?» |

Si partimos al azar por ensayo, los 100 ensayos de un mismo participante se reparten
entre entrenamiento y prueba. El modelo ve 80 ensayos de esa persona y se le pide
predecir los otros 20. Si el modelo tiene **con qué reconocer al individuo** —un
identificador de participante, efectos aleatorios por sujeto, o simplemente suficiente
flexibilidad—, entonces sabe de esa persona algo que no debería saber, y el resultado
sale optimista. Eso es **fuga de información** (*leakage*).

Ese margen no es pequeño: en el cuaderno 1 medimos que el 62 % de la varianza no
explicada es variación estable entre personas (ICC = 0.625). Todo eso es lo que puede
filtrarse. Cuánto se filtra de hecho depende del modelo, y en el ejercicio 1 vamos a
medirlo para el nuestro.

`GroupShuffleSplit` parte respetando los grupos: cada participante entero cae en
entrenamiento o en prueba, nunca en los dos. Un detalle: `test_size=0.20` se refiere a
la proporción de **grupos**, no de filas. Aquí coincide con el 20 % de los ensayos
porque los 30 participantes tienen exactamente 100 ensayos cada uno; con grupos de
tamaño desigual no coincidiría.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

variables = ['condition', 'trial', 'age']
X = data[variables]
y = data['RT_ms']
grupos = data['subject']          # <- la clave: agrupar por participante

particion = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
idx_train, idx_test = next(particion.split(X, y, groups=grupos))

X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]

print('Entrenamiento: %d ensayos, %d participantes' % (len(idx_train), grupos.iloc[idx_train].nunique()))
print('Prueba       : %d ensayos, %d participantes' % (len(idx_test), grupos.iloc[idx_test].nunique()))
print()
print('Participantes de prueba:', sorted(grupos.iloc[idx_test].unique()))
print('¿Se solapan los conjuntos?',
      len(set(grupos.iloc[idx_train]) & set(grupos.iloc[idx_test])) > 0)

Seis participantes (`S09`, `S10`, `S16`, `S18`, `S24`, `S28`) quedan enteramente fuera
del entrenamiento. El modelo aprenderá de los otros 24 y tendrá que predecir a estos
seis sin haber visto un solo ensayo suyo.

Esta es la situación real de cualquier aplicación clínica o de diagnóstico: llega un
paciente nuevo.

## 3. El pipeline

`condition` es texto y hay que convertirla en número; `trial` y `age` ya son numéricas
y pasan tal cual. `ColumnTransformer` aplica un tratamiento distinto a cada columna, y
`Pipeline` encadena ese preprocesamiento con el modelo.

Por qué esto importa y no es burocracia: al estar dentro de un `Pipeline`, el
codificador se ajusta **solo con los datos de entrenamiento** y luego se aplica al
conjunto de prueba. Si uno codificara a mano sobre la tabla completa, estaría dejando
que información del conjunto de prueba entre en la preparación.

Con dos categorías el riesgo es nulo, pero con una imputación por la media, un escalado
o una selección de variables la diferencia es real. Conviene ser preciso sobre lo que
`Pipeline` garantiza: protege el preprocesamiento **que contiene**, siempre que la
partición ocurra fuera de él. No protege de nada de lo que se haya hecho a los datos
antes —y, desde luego, no protege de haber partido mal la muestra, que es el tema de
este cuaderno.

`drop='first'` elimina una de las dos columnas indicadoras: con dos categorías basta una
(0 = congruente, 1 = incongruente), y mantener las dos haría el sistema redundante.

In [ ]:
preprocesamiento = ColumnTransformer([
    ('condition', OneHotEncoder(drop='first'), ['condition']),
    ('numeric',   'passthrough',               ['trial', 'age'])
])

modelo = Pipeline([
    ('preprocess', preprocesamiento),
    ('model',      LinearRegression())
])

modelo.fit(X_train, y_train)

coefs = modelo.named_steps['model'].coef_
print('Intercepto      : %8.2f' % modelo.named_steps['model'].intercept_)
print('Incongruencia   : %8.2f ms' % coefs[0])
print('Ensayo          : %8.2f ms/ensayo' % coefs[1])
print('Edad            : %8.2f ms/año' % coefs[2])

Los coeficientes son prácticamente los mismos que estimamos en el cuaderno 1 (+65 ms
de incongruencia, −0.58 ms por ensayo). **Es literalmente el mismo modelo lineal.**
Lo único que cambia es qué le preguntamos.

## 4. Evaluar sobre los participantes nuevos

In [ ]:
RT_predicho = modelo.predict(X_test)

rmse = mean_squared_error(y_test, RT_predicho) ** 0.5
r2   = r2_score(y_test, RT_predicho)

print('RMSE    = %.2f ms' % rmse)
print('R² test = %.4f' % r2)
print()
print('R² en entrenamiento = %.4f' % modelo.score(X_train, y_train))
print('Desv. estándar del RT = %.2f ms' % data['RT_ms'].std())

### El resultado, y por qué es el punto de la clase

| medida | valor |
|---|---|
| valor-p del efecto de condición | ~$10^{-110}$ |
| R² en participantes nuevos | **0.13** |
| RMSE | **80 ms** |
| desviación estándar del RT | 83 ms |

El RMSE, 80 ms, es casi la desviación estándar de la propia variable, 83 ms: el error de
predicción medio es casi tan grande como la variabilidad que había que explicar.
Comparémoslo explícitamente con el modelo más tonto posible: predecir siempre la media.

In [ ]:
import numpy as np

linea_base = np.full(len(y_test), y_train.mean())

print('Modelo    : RMSE %.2f ms | R² %.4f' % (
      mean_squared_error(y_test, RT_predicho) ** 0.5, r2_score(y_test, RT_predicho)))
print('Media sola: RMSE %.2f ms | R² %.4f' % (
      mean_squared_error(y_test, linea_base) ** 0.5, r2_score(y_test, linea_base)))

El modelo mejora sobre la media, pero poco: de 86 ms de error a 80 ms.

**Y no hay ninguna contradicción.** Las dos cosas son ciertas a la vez:

- Que la incongruencia añade ~65 ms *en promedio* es un hecho tan sólido como se puede
  desear: el modelo mixto lo sitúa entre 61.4 y 68.0 ms con un 95 % de confianza.
- Ese promedio explica poquísimo de **un ensayo concreto** de **una persona concreta**,
  porque el 62 % de lo que queda es quién es la persona, y a una persona nueva no la
  conocemos.

Vuelve al histograma solapado del cuaderno 1: eso ya se veía ahí. Dos distribuciones
tienen dos propiedades independientes —dónde están sus centros y cuánto se solapan— y
cada pregunta se fija en una. La significación habla de la **separación de los centros**
y mejora indefinidamente con el tamaño muestral; la calidad predictiva habla del
**solapamiento** y no mejora por medir a más gente.

Conviene descartar una explicación tentadora y equivocada: que el modelo **sobreajuste**
y por eso rinda mal fuera. No es eso. Con cuatro parámetros y 2400 filas no hay
sobreajuste posible, y se comprueba: partiendo al azar por ensayo, el R² de
entrenamiento es 0.191 y el de prueba 0.183, una brecha de 0.008. El modelo generaliza
perfectamente a ensayos nuevos.

Lo que no generaliza es a **personas** nuevas, y eso no es un defecto del ajuste sino de
los predictores. Con la condición, el ensayo y la edad no se llega más lejos, se valide
como se valide.

Una advertencia sobre el 0.13, por honestidad: depende bastante de qué seis
participantes toquen en el conjunto de prueba. Repitiendo la partición con 200 semillas
distintas, la mediana es 0.10 y la mitad central de los valores va de −0.03 a 0.15. Es
decir que, con estos predictores, predecir a una persona nueva es a menudo **igual o
peor que predecir la media**. El resultado con `random_state=42` es de los favorables.

La moraleja no es que la inferencia engañe. Es que responde a una pregunta distinta, y
que informar solo el valor-p deja al lector sin saber si el efecto sirve para algo a
nivel individual.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].scatter(RT_predicho, y_test, s=6, alpha=0.3, color='#4C72B0')
lims = [y_test.min() - 20, y_test.max() + 20]
ax[0].plot(lims, lims, 'k--', lw=1, label='predicción perfecta')
ax[0].set_xlabel('RT predicho (ms)'); ax[0].set_ylabel('RT observado (ms)')
ax[0].set_title('Participantes nuevos: predicho vs. observado'); ax[0].legend()

residuos = y_test.values - RT_predicho
ax[1].hist(residuos, bins=40, color='#C44E52', alpha=0.75)
ax[1].axvline(0, color='k', lw=1)
ax[1].set_xlabel('residuo: observado − predicho (ms)'); ax[1].set_ylabel('ensayos')
ax[1].set_title('Error de predicción')

plt.tight_layout(); plt.show()

El gráfico de la izquierda es elocuente: las predicciones se apiñan en dos columnas
estrechas —una por condición— mientras los valores observados se extienden por
centenares de milisegundos. El modelo solo sabe decir «congruente» o «incongruente»,
y dentro de cada columna está adivinando.

> ### ✋ TU TURNO — 1
>
> Repite el ajuste partiendo la muestra **al azar por ensayo** en vez de por
> participante:
>
> ```python
> from sklearn.model_selection import train_test_split
> X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
> ```
>
> Ajusta el mismo pipeline y compara el R² de prueba con el 0.13 de arriba.
>
> - ¿Sube o baja?
> - ¿Qué está midiendo esa segunda cifra, exactamente?
> - ¿Cuál de las dos reportarías en un artículo, y qué tendrías que escribir al lado?

In [ ]:
# Escribe aquí tu respuesta.



> ### ✋ TU TURNO — 2
>
> Intenta darle al modelo la variable que de verdad le falta: el participante.
>
> ```python
> X2 = data[['condition', 'trial', 'age', 'subject']]
> ```
>
> y añade `subject` al `OneHotEncoder`. Ajusta con la partición **por participante**
> (`idx_train` / `idx_test`) y observa qué pasa.
>
> - ¿Qué error da, o qué predicción produce?
> - ¿Por qué era inevitable?
>
> *Pista: `OneHotEncoder(drop='first', handle_unknown='ignore')` evita que reviente y
> deja ver qué hace el modelo con una categoría que nunca vio.*

In [ ]:
# Escribe aquí tu respuesta.



---

## 5. Hacia dónde sigue esto

El techo que acabamos de tocar tiene una causa identificable: nuestras variables
describen el **estímulo** (congruente o no, en qué momento de la sesión) y casi nada de
la **persona**. La edad es lo único individual que tenemos, y en esta muestra no
predice nada: +0.28 ms por año, con un intervalo que va de −3.7 a +4.2 (p = 0.89 según
el modelo mixto, que es el que cuenta bien las 30 observaciones independientes de edad
que tenemos; el OLS sobre 3000 filas da p = 0.26 porque subestima el error estándar en un
factor de ocho). El rango de 21 a 40 años, además, es estrecho.

De ahí sale la propuesta de la última parte de la clase: meter una variable del
cerebro entre la condición y la conducta.

$$\text{Condición} \;\rightarrow\; \text{Conectividad cerebral} \;\rightarrow\; RT / \text{Aciertos}$$

Modelado, son dos ecuaciones encadenadas:

$$FC_{ij} = \alpha_0 + \alpha_1 \text{Condición}_{ij} + v_j + \eta_{ij}$$
$$RT_{ij} = \beta_0 + \beta_1 \text{Condición}_{ij} + \beta_2 FC_{ij} + u_j + \epsilon_{ij}$$

Lo que cambia respecto de hoy es que $FC_{ij}$ **sí varía dentro de la persona y entre**
**personas**: es una medida individual, disponible ensayo a ensayo. Es el tipo de
variable que podría rescatar parte de ese 62 % inaccesible.

Y hay una segunda razón, distinta de la predictiva, para querer esa variable: una
cadena `Condición → Cerebro → Conducta` se propone como hipótesis **mecanicista**, no
solo como un predictor mejor. Aspira a decir *por dónde* pasa el efecto.

Aspira, con cuidado: ajustar esas dos regresiones no demuestra un mecanismo. La
condición está aleatorizada, pero $FC$ no lo está, así que $\beta_2$ solo se lee
causalmente si suponemos que no hay nada que influya a la vez sobre la conectividad y
sobre el RT. Ese supuesto no se comprueba con estos datos: se argumenta, o se pone a
prueba interviniendo. Esa distinción —entre un modelo que predice, uno que ajusta y uno
que explica— es la que atraviesa el curso entero.

### Los tres niveles que se complementan

| nivel | forma |
|---|---|
| conductual | Condición → Conducta |
| mecanicista | Condición → Actividad cerebral → Conducta |
| predictivo | Actividad cerebral + Condición → Conducta |

---

## Para llevarse

1. **Inferencia y predicción responden preguntas distintas.** El mismo modelo lineal,
   sobre los mismos datos, da $p \approx 10^{-110}$ y $R^2 = 0.13$. Ninguna de las dos
   cifras es un error.
2. **Cómo se parte la muestra es parte de la pregunta.** Partir por ensayo responde
   «¿otro ensayo de alguien que conozco?»; partir por participante responde «¿una
   persona nueva?».
3. **La varianza entre personas es real y a la vez inaccesible** si no medimos nada de
   la persona nueva. El ICC de 0.62 y el R² de 0.13 son la misma observación vista
   desde dos lados.
4. **Un efecto significativo no es un efecto útil**, y un modelo predictivo bueno no
   explica nada por sí solo. Hacen falta los dos, y hacen falta por razones distintas.

---

## Una última cosa: estos datos son simulados

Se generaron con una estructura muy parecida a la que hemos estado ajustando: un
efecto de condición, un efecto lineal de ensayo, un desplazamiento por participante
sacado de una normal y ruido normal encima; y los aciertos, aparte, con una
probabilidad que depende solo de la condición.

Por eso todo encajó tan limpiamente. Hemos recuperado el modelo que generó los datos,
que es un ejercicio útil pero distinto de estudiar un fenómeno. Merece la pena saber
en qué se notaría la diferencia, porque son cosas que se pueden comprobar en
cualquier conjunto de datos:

- **La forma de la distribución.** Aquí el RT es simétrico. En datos humanos es
  claramente asimétrico hacia la derecha, con una cola de ensayos lentos, y se
  modela con distribuciones ex-gaussianas en lugar de normales.
- **Las colas.** Aquí no hay ninguna respuesta por debajo de 250 ms ni por encima de
  1000. En datos reales hay anticipaciones y despistes, y decidir qué hacer con ellos
  es una de las primeras decisiones del análisis.
- **La relación entre errores y latencia.** Aquí los ensayos incorrectos tardan lo
  mismo que los correctos. En datos reales los errores suelen ser marcadamente más
  rápidos, porque acierto y latencia salen del mismo proceso de decisión.
- **Las diferencias individuales en precisión.** Aquí los 30 participantes aciertan
  con la misma probabilidad; la variación que se observa es puro azar binomial. El RT,
  en cambio, sí tiene un efecto de participante grande (el ICC de 0.62). Esa asimetría
  es una decisión de quien generó los datos, no algo que ocurra en la naturaleza.
- **La estructura secuencial.** Aquí un ensayo no informa sobre el siguiente. En datos
  reales la atención fluctúa lentamente, la gente se enlentece después de un error, y
  la interferencia depende de si el ensayo anterior era congruente.

Ninguna de esas ausencias afecta a lo que hicimos hoy. Todas afectarían a un análisis
de datos reales, y varias de ellas son, en sí mismas, fenómenos que vale la pena
estudiar.